# EDA — Enterprise Data Architecture & Financial Reconciliation

Exploratory analysis on the reconciled dataset produced by `istudio_clean.ipynb`
(`final_reconciled_data.csv` — HR user status joined with cleaned server
transaction logs and forward-filled EUR/USD exchange rates).

**Columns:** `Date`, `User_ID`, `Product_ID`, `Euro_Value`, `Exchange_Rate`,
`USD_Revenue`, `name`, `status`, `updated_at`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_csv("final_reconciled_data.csv", parse_dates=["Date"])
df.shape

## 1. First look

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 2. Data quality checks

In [ ]:
# Missing values
df.isna().sum()

In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate transactions (User_ID + Date + Product_ID):",
      df.duplicated(subset=["User_ID", "Date", "Product_ID"]).sum())

In [ ]:
# Date range and status breakdown
print("Date range:", df["Date"].min(), "to", df["Date"].max())
df["status"].value_counts()

## 3. Revenue distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["USD_Revenue"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("USD Revenue Distribution")

sns.boxplot(x=df["USD_Revenue"], ax=axes[1])
axes[1].set_title("USD Revenue — Outlier Check")
plt.tight_layout()
plt.show()

In [ ]:
# Outlier snapshot using IQR
q1, q3 = df["USD_Revenue"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outliers = df[df["USD_Revenue"] > upper]
print(f"Upper IQR bound: {upper:,.2f}")
print(f"Outlier transactions: {len(outliers)} ({len(outliers)/len(df):.2%} of rows)")
outliers.sort_values("USD_Revenue", ascending=False).head(10)

## 4. Trends over time

In [ ]:
monthly = df.set_index("Date").resample("ME")["USD_Revenue"].agg(["sum", "count"])
monthly.columns = ["Total_Revenue", "Transaction_Count"]

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly.index, monthly["Total_Revenue"], marker="o", color="steelblue", label="Revenue (USD)")
ax1.set_ylabel("Total USD Revenue", color="steelblue")
ax1.set_xlabel("Month")

ax2 = ax1.twinx()
ax2.bar(monthly.index, monthly["Transaction_Count"], alpha=0.2, width=15, color="gray", label="Transactions")
ax2.set_ylabel("Transaction Count", color="gray")

plt.title("Monthly Revenue and Transaction Volume")
fig.tight_layout()
plt.show()

In [ ]:
monthly

## 5. Exchange rate behavior

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
rate_by_date = df.groupby("Date")["Exchange_Rate"].mean()
ax.plot(rate_by_date.index, rate_by_date.values, color="darkorange")
ax.set_title("EUR/USD Exchange Rate Over Time (forward-filled weekends)")
ax.set_ylabel("Exchange Rate")
plt.tight_layout()
plt.show()

## 6. Top products and users

In [ ]:
top_products = (
    df.groupby("Product_ID")["USD_Revenue"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "Total_Revenue", "count": "Transactions"})
    .sort_values("Total_Revenue", ascending=False)
    .head(15)
)
top_products

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=top_products["Total_Revenue"], y=top_products.index, ax=ax, palette="viridis")
ax.set_title("Top 15 Products by USD Revenue")
ax.set_xlabel("Total USD Revenue")
plt.tight_layout()
plt.show()

In [ ]:
top_users = (
    df.groupby("User_ID")["USD_Revenue"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "Total_Revenue", "count": "Transactions"})
    .sort_values("Total_Revenue", ascending=False)
    .head(15)
)
top_users

## 7. Active vs inactive user contribution

In [ ]:
status_revenue = df.groupby("status")["USD_Revenue"].agg(["sum", "mean", "count"])
status_revenue.columns = ["Total_Revenue", "Avg_Revenue", "Transactions"]
status_revenue

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(x=status_revenue.index, y=status_revenue["Total_Revenue"], ax=ax, palette="Set2")
ax.set_title("Total USD Revenue by User Status")
plt.tight_layout()
plt.show()

## 8. Correlations

In [ ]:
num_cols = ["Euro_Value", "Exchange_Rate", "USD_Revenue"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.show()

## 9. Key takeaways

_Fill in after reviewing the outputs above — e.g. seasonality in monthly
revenue, concentration of revenue in top products/users, any residual
outliers worth flagging to the reconciliation process, and whether
Active vs Inactive status meaningfully affects revenue contribution._
